# Visualizing Spatially Variable Genes with Squidpy

Reproducible code for the blog post at [spatiabio.com](https://www.spatiabio.com).

Dataset: 10x Genomics Visium mouse brain demo. This notebook visualizes the top 5 SVGs identified in the [Moran's I post](https://www.spatiabio.com/2026/06/spatially-variable-genes-with-squidpy.html).

## Setup

In [ ]:
import squidpy as sq
import scanpy as sc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

## Load and preprocess

Same pipeline as the SVG analysis post.

In [ ]:
adata = sq.datasets.visium_hne_adata()
# Or load from file: adata = sc.read_h5ad("visium_hne_adata.h5ad")

sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
# NOTE: visium_hne_adata() ships .X ALREADY log-normalized (max ~8.4, non-integer);
# the raw integer counts live in adata.raw.X. Calling normalize_total + log1p here
# would log it a second time -- silently, with no error, changing every result below.
sq.gr.spatial_neighbors(adata)
print(f"Shape: {adata.shape}")

## Visualize top 5 SVGs

Top genes from Moran's I analysis (post 5), corrected pipeline:

| Rank | Gene | Moran's I |
|------|------|-----------|
| 1 | Prkcd | 0.824 |
| 2 | Pmch | 0.818 |
| 3 | Mobp | 0.811 |
| 4 | Agrp | 0.799 |
| 5 | Tcf7l2 | 0.794 |

In [ ]:
top5 = ["Prkcd", "Pmch", "Mobp", "Agrp", "Tcf7l2"]

sq.pl.spatial_scatter(
    adata,
    color=top5,
    ncols=3,
    title=[f"{g}" for g in top5]
)

## Individual gene plots

In [ ]:
for gene in top5:
    sq.pl.spatial_scatter(
        adata,
        color=gene,
        title=f"{gene} (Moran's I spatial expression)"
    )

## Fix colormap scale for cross-gene comparison

> By default each gene scales independently, so colours are not comparable between panels. Pass `vmin` / `vmax` to pin the same range across genes.

In [ ]:
sq.pl.spatial_scatter(
    adata, color="Prkcd", use_raw=False,
    vmin=0, vmax=8,   # .X is log-normalized; across these 5 genes the max is ~7.9.
                      # (the old vmax=20 was off the scale, washing every plot out)
    title="Prkcd (fixed scale)"
)

## Pattern summary

The corrected top 5 are dominated by **regional** and **tract** patterns rather than laminar ones:

**Regional** (compact clusters = discrete anatomical structures):
- `Prkcd`, `Tcf7l2` — thalamus
- `Pmch` — hypothalamus
- `Agrp` — hypothalamic arcuate nucleus; very focal, near-zero elsewhere (mean 0.05)

**Tract** (long myelinated bundles):
- `Mobp` — white matter / fiber tracts; the most broadly expressed of the five (mean 2.42)

Both score high on Moran's I but for different reasons — a tract gene like `Mobp` has many
same-signal neighbours along the bundle, while a focal gene like `Agrp` has extreme on/off
contrast against a silent background.